## Simulate SNIa with Opsim

- **Author:** Sylvie Dagoret-Campagne
- **Affiliation:** : IJCLab/IN2P3/CNRS
- **Creation date:** : 2026-07-25
- **Last update** : 2026-08-11 
- **mac**: python kernel = conda_py313

https://skysurvey.readthedocs.io/en/latest/howto/load_lsst.html

## Import

In [ ]:
import skysurvey
import os

## Configuration

In [ ]:
# RUBIN_SIM_DATA_DIR points to the local cache of rubin_sim/rubin_scheduler auxiliary data
# (opsim databases, dust maps, SN gamma/noise files, throughputs, ...).
# os.environ["RUBIN_SIM_DATA_DIR"] = "/users/dagoret/DATA/OpSim"
PATH_OPSIM = os.getenv("RUBIN_SIM_DATA_DIR")
print(f"PATH_OPSIM = {PATH_OPSIM}")

In [ ]:
file_opsim = "sim_baseline/baseline_v5.3.5_10yrs.db"
# file_opsim = "ddf_one_less_v5.3.2_10yrs.db"
# file_opsim = "ddf_sd_v5.3.0_10yrs.db"

In [ ]:
# lsst opsim files are large, this may take a few minutes
opsim_path = os.path.join(PATH_OPSIM, file_opsim)
lsst = skysurvey.LSST.from_opsim(opsim_path)

Note:

    Depending on the version of your opsim file, some columns names in the logs may differ. skysurvey handles this automatically.

Survey properties

The survey data contains one row per observation (pointing), with the standard skysurvey columns: mjd, band, skynoise, gain, zp, plus LSST-specific ones like observationId:

In [ ]:
lsst.data

The survey spans ~10 years, from the start of LSST operations. We can check the exact date range:

- `bands` must be renamed in order to remove numbers:

In [ ]:
lsst.data["band"] = lsst.data["band"].str.split("_").str[0]

In [ ]:
lsst.data

## Explore opsim

In [ ]:
from astropy.time import Time

tmin, tmax = lsst.date_range
print(f"Start: {Time(tmin, format='mjd').iso}")
print(f"End:   {Time(tmax, format='mjd').iso}")
print(f"Duration: {(tmax - tmin)/365.25:.1f} years")

The LSST camera has a roughly circular focal plane with a cross-shaped chip arrangement (3-5-5-5-3 CCD structure), covering ~9.6 deg². This is the single-field footprint used to match sky positions to observations:

In [ ]:
fig = lsst.show_footprint(add_text=True)

We can easily check which filter bands are included in the specific database we’re using with:

In [ ]:
print(f"Bands covered: {sorted(lsst.data['band'].unique())}")

We can visualize the observing cadence (the number of exposures per day), broken down by band. This shows the survey strategy and gaps due to the weather, maintenance, or seasonal gaps.

In [ ]:
fig = lsst.show_nexposures(exposure_key="observationId")  # observationId is LSST's specific exposure_key

Loading specific subsets

The full 10-year opsim is large (~3M rows). You can load a subset using the sql_where argument, which accepts any valid SQLite WHERE.

For example, to load the 1rst year of LSST:

In [ ]:
lsst_yr1 = skysurvey.LSST.from_opsim(opsim_path, sql_where="night<365")

In [ ]:
fig = lsst_yr1.show_nexposures(exposure_key="observationId")  # observationId is LSST's specific exposure_key

or just select the u-band:

In [ ]:
lsst_y = skysurvey.LSST.from_opsim(opsim_path, sql_where="band IN ('y')")

In [ ]:
fig = lsst_y.show_nexposures(exposure_key="observationId")  # observationId is LSST's specific exposure_key

You can also combine conditions with AND/OR, filter by band, night, field, etc. For example, to load only the first year excluding the y-band:

In [ ]:
lsst_no_y_yr1 = skysurvey.LSST.from_opsim(
    opsim_path, sql_where="band IN ('u', 'g', 'r', 'i', 'z') AND night < 365"
)

In [ ]:
fig = lsst_no_y_yr1.show_nexposures(
    exposure_key="observationId"
)  # observationId is LSST's specific exposure_key

We can visualize the sky coverage for the 1rst year of LSST:

In [ ]:
# 5 bands, year 1
coverage_yr1 = lsst_yr1.get_fieldcoverage()
lsst_yr1.show(vmin=0, vmax=coverage_yr1.quantile(0.95), cmap="coolwarm")

And compare it to the entire duration of the survey:

In [ ]:
# 6 bands, 10 years
coverage = lsst.get_fieldcoverage()
lsst.show(vmin=0, vmax=coverage.quantile(0.95), cmap="coolwarm")

- *get_fieldcoverage()* returns the number of observations per field.
- Passing it to *show()* via *vmax* controls the color scale. In this example, we use the 95th percentile to enhance the constrast between fields that receive a high number of visits (e.g. deep drilling fields, overlap regions) and less-covered regions.

## Simulate SNe Ia with LSST:

With the survey loaded, we can simulate SNe Ia as they would be observed by LSST. We match the survey time range and set a redshift limit, and simulate 10 000 SNe Ia:

In [ ]:
tmin, tmax = lsst.date_range
snia = skysurvey.SNeIa.from_draw(size=10_000, tstart=tmin, tstop=tmax, zmax=0.5)

We generate the observed light curves by matching each SN’s position to the LSST pointings. Only a fraction of the 10 000 simulated SNe Ia have light curves, as objects falling outside the survey footprint are automatically excluded.

In [ ]:
dset = skysurvey.DataSet.from_targets_and_survey(snia, lsst, progress_bar=True, discard_bands=True)

Note on `discard_bands=True`: Observations in bands whose wavelength range falls outside the spectral coverage of the defined sncosmo model are automatically removed.

This can happen in two cases:

    Band redder than model spectral range (low redshift): the band’s red edge exceeds the model’s maximum wavelength. This occurs, for example, for LSST’s lssty band (9084–10945 Å) when using the SALT2 model at low redshift (z ≲ 0.19), where the model has not yet been sufficiently redshifted to cover the y-band.

    Band bluer than model spectral range (high redshift): the band’s blue edge falls below the model’s observer-frame minimum wavelength, which shifts to longer wavelengths as redshift increases. This can affect the lsstu band (3105-4086 Å) at high redshift. In simulations extending to z ≈ 2.5, this effect starts to appear around z ≳ 0.6-0.8, becomes common for z ≳ 1.5, and dominates at z ≳ 2, where a significant fraction of observations can be discarded for a given SN. Overall, ~60–80% of simulated SNe Ia have at least one discarded observation due to this effect, with the number of discarded observations per SN varies widely (from a few points to several tens depending on redshift).

In this notebook (z ≤ 0.5), only the first case occurs, affecting a small fraction of the sample (~3–5% of SNe Ia, in the y-band, at low redshift (z ≲ 0.19)).

In [ ]:
number_of_detections = dset.get_ndetection()

In [ ]:
number_of_detections

In [ ]:
idx_max = number_of_detections.idxmax()
max_detections = number_of_detections.max()

print(f"Index : {idx_max}")
print(f"Number of detection : {max_detections}")

*get_ndetection()*  returns the number of 5σ detections per SN Ia. SNe Ia with zero detections are excluded from dset.data.

The light curves of a specific SN Ia shows observations in all LSST bands, for the phase range [-20, +40] days:

In [ ]:
fig = dset.show_target_lightcurve(index=idx_max, phase_window=[-20, 40])
fig.legend({"legend": True})
fig.show()

Quality cuts can then be applied to our sample to select SNe Ia with good light curves sampling for a cosmological analysis. A more detailed notebook, “Realistic use of the LSST survey” will be added in the documentation to “Analysis examples” section.

https://skysurvey.readthedocs.io/en/latest/howto/load_lsst.html

In [ ]:
%pinfo dset.show_target_lightcurve